# Week 3 Entity Extraction Evaluation

This notebook reviews the Week 3 entity extraction artifacts, compares the rule-based extractor with the saved NER model, and uses a evaluation set for the error analysis.


In [1]:
from collections import Counter
import json
import sys

import pandas as pd

sys.path.append('..')

from src.real_estate_nlp.entity_extractor import EntityExtractor
from scripts.evaluate_entities import SpacyNerExtractor, evaluate

---

## 1. Artifacts Loading

Load the labeled datasets, taxonomy, extractor, evaluation helpers, and the saved NER model.

In [2]:
with open('../data/processed/entity_train_labels.json') as f:
    train_labels = json.load(f)['items']

with open('../data/processed/entity_dev_labels.json') as f:
    dev_labels = json.load(f)['items']

with open('../data/processed/entity_eval_labels.json') as f:
    eval_labels = json.load(f)['items']

with open('../data/processed/taxonomy.json') as f:
    taxonomy = json.load(f)

rule_extractor = EntityExtractor(taxonomy_path='../data/processed/taxonomy.json')
ner_extractor = SpacyNerExtractor('../data/models/entity_ner')
hybrid_extractor = EntityExtractor(
    taxonomy_path='../data/processed/taxonomy.json',
    ner_model=ner_extractor,
)

---

## 2. Label Set Profile

Before scoring extraction methods, check the size and label mix of the reviewed data. The held-out set is the main evaluation target.

In [3]:
def label_count(items):
    return sum(len(item['entities']) for item in items)

pd.DataFrame([
    {'split': 'train', 'remarks': len(train_labels), 'entities': label_count(train_labels)},
    {'split': 'dev', 'remarks': len(dev_labels), 'entities': label_count(dev_labels)},
    {'split': 'eval', 'remarks': len(eval_labels), 'entities': label_count(eval_labels)},
])

,split,remarks,entities
0,train,640,13855
1,dev,160,3568
2,eval,200,2619


In [4]:
def label_distribution(items):
    counts = Counter(entity['label'] for item in items for entity in item['entities'])
    total = sum(counts.values())
    return pd.DataFrame(
        [{'label': label, 'count': count, 'pct': count / total} for label, count in counts.items()]
    ).sort_values('count', ascending=False)

label_distribution(eval_labels).assign(pct=lambda df: df['pct'].round(3))

,label,count,pct
9,room,441,0.168
10,interior_feature,438,0.167
4,location,417,0.159
5,condition,265,0.101
7,exterior_feature,202,0.077
2,bedrooms,160,0.061
3,bathrooms,139,0.053
12,amenity,119,0.045
11,parking,110,0.042
6,property_type,97,0.037


The evaluation set is intentionally broad. The largest buckets are text-heavy labels such as `interior_feature`, `room`, and `location`, which are harder than numeric facts.

In [5]:
pd.Series({
    'taxonomy_terms': len(taxonomy['terms']),
    'taxonomy_categories': len(set(term['category'] for term in taxonomy['terms'])),
}).to_frame('count')

,count
taxonomy_terms,304
taxonomy_categories,8


---

## 3. Method Comparison

Score the rule extractor, the saved NER model, and the hybrid extractor on the same held-out labels.

Metric suffixes used in the comparison table:

- `rule_strict`: rule-based extractor scored by exact label, value, and span.
- `rule_overlap`: rule-based extractor scored when the predicted span overlaps the gold span with the same label and value.
- `rule_value`: rule-based extractor scored by label and normalized value only; span boundaries are ignored.
- `span`: model output scored by label and exact span only; value matching is not used.

In [6]:
systems = {
    'rule_strict': (rule_extractor, 'strict'),
    'rule_overlap': (rule_extractor, 'overlap'),
    'rule_value': (rule_extractor, 'value'),
    'ner_span': (ner_extractor, 'span'),
    'hybrid_span': (hybrid_extractor, 'span'),
}

results = {
    name: evaluate(eval_labels, extractor, match_mode)
    for name, (extractor, match_mode) in systems.items()
}

summary = pd.DataFrame([
    {'system': name, **result['overall']}
    for name, result in results.items()
])

summary[['system', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn']].round(3)

,system,precision,recall,f1,tp,fp,fn
0,rule_strict,0.889,0.814,0.850,2133,266,486
1,rule_overlap,0.916,0.839,0.876,2197,202,422
2,rule_value,0.925,0.848,0.885,2220,179,399
3,ner_span,0.822,0.727,0.772,1903,411,716
4,hybrid_span,0.877,0.831,0.853,2177,306,442


The rule extractor is still the strongest practical baseline. NER has a strong dev score but loses recall on the held-out labels. The hybrid version recovers some recall, but the precision tradeoff keeps F1 flat.

In [7]:
per_label = pd.DataFrame([
    {'label': label, **row}
    for label, row in results['rule_strict']['per_label'].items()
]).sort_values('f1')

per_label[['label', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn']].round(3)

,label,precision,recall,f1,tp,fp,fn
5,hoa_fee,0.750,0.750,0.750,6,2,2
0,amenity,0.764,0.815,0.789,97,30,22
7,location,0.838,0.755,0.794,315,61,102
3,condition,0.870,0.755,0.808,200,30,65
11,room,0.890,0.751,0.814,331,41,110
4,exterior_feature,0.872,0.777,0.822,157,23,45
9,parking,0.848,0.809,0.828,89,16,21
10,property_type,0.914,0.763,0.831,74,7,23
14,transaction_or_listing,0.922,0.816,0.866,71,6,16
6,interior_feature,0.906,0.854,0.879,374,39,64


The weakest strict scores are mostly broad language categories. Numeric facts are much cleaner because their surface forms are easier to normalize and match.

---

## 4. Extraction Examples

A few examples make the method differences easier to inspect than aggregate metrics alone.

In [8]:
examples = [
    'updated 3 bedroom 2 bathroom condo with community pool and attached garage',
    'single story home on 0.25 acre lot with remodeled kitchen and ocean views',
    'seller financing available for duplex with rental income and private patio',
]

rows = []
for text in examples:
    for method, extractor in [('rule', rule_extractor), ('ner', ner_extractor), ('hybrid', hybrid_extractor)]:
        entities = extractor.extract_all(text)
        rows.append({
            'text': text,
            'method': method,
            'entities': [(e['label'], e['value'], e['text']) for e in entities],
        })

pd.DataFrame(rows)

,text,method,entities
0,updated 3 bedroom 2 bathroom condo with commun...,rule,"[(condition, updated, updated), (bedrooms, 3, ..."
1,updated 3 bedroom 2 bathroom condo with commun...,ner,"[(condition, updated, updated), (bedrooms, 3 b..."
2,updated 3 bedroom 2 bathroom condo with commun...,hybrid,"[(condition, updated, updated), (bedrooms, 3, ..."
3,single story home on 0.25 acre lot with remode...,rule,"[(stories, 1, single story), (lot_size, 0.25, ..."
4,single story home on 0.25 acre lot with remode...,ner,"[(stories, single story, single story), (lot_s..."
5,single story home on 0.25 acre lot with remode...,hybrid,"[(stories, 1, single story), (lot_size, 0.25, ..."
6,seller financing available for duplex with ren...,rule,"[(transaction_or_listing, seller financing, se..."
7,seller financing available for duplex with ren...,ner,"[(transaction_or_listing, seller financing, se..."
8,seller financing available for duplex with ren...,hybrid,"[(transaction_or_listing, seller financing, se..."


The rule extractor is more predictable on numeric facts and taxonomy terms. NER is useful as a comparison point, but its held-out behavior is less stable.

---

## 5. Error Analysis

Start with the rule extractor because it is the current path. The goal is to find failure patterns that are worth fixing.

In [9]:
def error_frame(result, kind):
    return pd.DataFrame(result[kind])

rule_fp = error_frame(results['rule_strict'], 'false_positives')
rule_fn = error_frame(results['rule_strict'], 'false_negatives')

pd.DataFrame({
    'false_positive': rule_fp['label'].value_counts(),
    'false_negative': rule_fn['label'].value_counts(),
}).fillna(0).astype(int).sort_values('false_negative', ascending=False)

,false_positive,false_negative
label,,
room,41,110
location,61,102
condition,30,65
interior_feature,39,64
exterior_feature,23,45
property_type,7,23
amenity,30,22
parking,16,21
transaction_or_listing,6,16


False negatives show where recall is weak. False positives show where taxonomy or regex matches are too broad.

In [10]:
rule_fn[['label', 'value', 'text', 'context']].head(12)

,label,value,text,context
0,room,bathroom,bathroom,welcome to this beautiful completely remodeled...
1,location,close to shopping,close to shopping,welcome to this beautiful completely remodeled...
2,room,bedroom,bedrooms,lovely exterior. interior dated and needs comp...
3,room,bedroom,bedroom,lovely exterior. interior dated and needs comp...
4,exterior_feature,backyard,backyard,"directions, take california city blvd. go sout..."
5,condition,cozy,cozy,seller says bring all offers 4 bedroom 2 bathr...
6,location,lake,lake,seller says bring all offers 4 bedroom 2 bathr...
7,parking,parking,parking,discover this charming mountain getaway at 620...
8,year_built,1922,1922 farm house,priced to sell.charming cherryland 2 story 192...
9,exterior_feature,patio,patio,priced to sell.charming cherryland 2 story 192...


The first misses usually point to coverage gaps: phrasing not in the taxonomy, implicit real estate language, or spans that are longer than the extractor currently emits.

In [11]:
rule_fp[['label', 'value', 'text', 'context']].head(12)

,label,value,text,context
0,condition,upgraded,upgraded,"nestled in the hills of alpine, this charming ..."
1,room,living room,living area,move in ready updated 1 story w rare to find t...
2,interior_feature,island,island,pride of ownership throughout. originally year...
3,location,dining,dining,va approved charming upstairs end unit condo w...
4,exterior_feature,yard,yard,just hit the market in north bakersfield this ...
5,condition,renovation,renovation,"well maintained 4 bedroom, 2 bath single famil..."
6,condition,freshly painted,interior paint,located in one of north county's most desirabl...
7,interior_feature,floor plan,functional layout,"3 bedroom, 2 bathroom home with tons of potent..."
8,location,transit,transit,great opportunity a prime transit oriented hig...
9,parking,garage,garage,welcome to 12564 eckleson st. enjoy the conven...


The false positives are useful for tightening rules. Common causes are generic terms, phrase boundaries, and words that shift meaning depending on context.